In [1]:
import os
from tqdm.notebook import tqdm
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
import codecs
from langchain_core.documents import Document

C:\Users\DHRUV\AppData\Local\Temp\ipykernel_18928\1150334825.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [2]:
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

from uuid import uuid4

In [3]:
# load .txt files from directory
data_dir = "data"
docs = []
for file in os.listdir(data_dir):
    if file.endswith(".txt") and not file.startswith("malicious_"):
        path = f"{data_dir}/{file}"
        with codecs.open(path, "r", encoding="utf-8", errors="ignore") as f:
            text = f.read()
        docs.append(Document(page_content=text, metadata={"source": path}))

In [4]:
# split text into chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=1024, chunk_overlap=128)
all_splits = [split for doc in docs for split in splitter.split_documents([doc])]

In [5]:
len(all_splits)

72

Use Huggingface Embeddings: [multilingual-e5-small](https://huggingface.co/intfloat/multilingual-e5-small)

In [6]:
model_name="intfloat/multilingual-e5-small"
embed_model = HuggingFaceEmbeddings(model_name=model_name, cache_folder="cached_models/")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [7]:
# generate embeddings with progress bar
# embeddings = [embed_model.embed_documents([chunk.page_content]) for chunk in tqdm(all_splits, desc="Embedding Chunks")]

In [8]:
index = faiss.IndexFlatL2(len(embed_model.embed_query("sample query")))

In [9]:
vector_store = FAISS(
    embedding_function=embed_model,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

In [10]:
uuids = [str(uuid4()) for _ in range(len(all_splits))]

In [11]:
vector_store.add_documents(documents=all_splits, ids=uuids)

['86a7ead8-00af-4943-9482-27220e0badef',
 'a513e426-98e4-4679-baeb-af8d87b7f42d',
 'f6fdf3d3-071d-4f89-ac23-b11c9d5b6c03',
 'c19bfa59-cbf8-44a3-806c-ac1beaeb2bbd',
 'e9a382f4-6a07-4d10-9896-05be0f1c80c4',
 '1a6ebd48-9d0f-4ab8-8f01-92144216a123',
 'ef23465b-4046-4ec0-9b78-9143bac9dc8c',
 '8e39a75a-49cd-45f6-a194-af7ade0406c2',
 'f9fc6895-2714-4dbe-bb7d-fad604299e84',
 'ddb12263-81ab-452f-96ee-6d61fe2e21fb',
 'bee5139b-73d2-4df1-9bbf-148f520e7115',
 'e80b8c88-21c5-49af-8033-3c65c2e7ba6c',
 'de93574e-8de0-4cca-8c49-23f13cf48f52',
 'deabc703-f853-41b3-a5c8-415f8d49c19f',
 '09a97f7b-339c-4266-9275-cb5b0eed47e1',
 'f76b05b6-8179-45a5-a323-3566f86e5779',
 'a7920672-d10c-4a56-80fe-05ae1c2459a9',
 '07695c70-2619-4e79-b624-6b34c4a9cf8a',
 'edec59d9-b54e-479f-a204-48396694f448',
 '2a1974c8-dac6-49d8-8f9c-4c3afc9a9a14',
 'b7a69af8-c6cb-451b-8dc6-1c48f133bb6a',
 '1abfd92b-74c3-4610-804c-5cc5cbf21727',
 'a57b4196-ceb7-45c8-b13c-fa20694d45c9',
 'd6e23258-26ce-4d33-945a-0f4ec9831e5b',
 '0f9571ed-d8ad-

Query

In [12]:
results = vector_store.similarity_search(
    "What is the core of quantum computing?",
    k=2,
)
for res in results:
    print(f"* {res.page_content} [{res.metadata}]")

* It is a branch of science that encompasses a broad range of methodologies from various disciplines, such as biochemistry, biotechnology, biomaterials, material science/engineering, genetic engineering, molecular biology, molecular engineering, systems biology, membrane science, biophysics, chemical and biological engineering, electrical and computer engineering, control engineering and evolutionary biology.

It includes designing and constructing biological modules, biological systems, and biological machines, or re-designing existing biological systems for useful purposes.[2]

Additionally, it is the branch of science that focuses on the new abilities of engineering into existing organisms to redesign them for useful purposes.[3] [{'source': 'data/Synthetic_biology.txt'}]
* PCs have periodic inclusions that inhibit wave propagation due to the inclusions' destructive interference from scattering. The photonic bandgap property of PCs makes them the electromagnetic analog of electronic

In [13]:
results = vector_store.similarity_search(
    "the most important technological innovation in history",
    k=5,
)
for res in results:
    print(f"* {res.page_content} [{res.metadata}]")
    print("="*10, "end", "="*10)

* Articles about Electromagnetism  Electricity Magnetism Optics History Computational Textbooks Phenomena Electrostatics Charge density Conductor Coulomb law Electret Electric charge Electric dipole Electric field Electric flux Electric potential Electrostatic discharge Electrostatic induction Gauss law Insulator Permittivity Polarization Potential energy Static electricity Triboelectricity Magnetostatics Ampère law Biot–Savart law Gauss magnetic law Magnetic dipole Magnetic field Magnetic flux Magnetic scalar potential Magnetic vector potential Magnetization Permeability Right-hand rule Electrodynamics Bremsstrahlung Cyclotron radiation Displacement current Eddy current Electromagnetic field Electromagnetic induction Electromagnetic pulse Electromagnetic radiation Faraday law Jefimenko equations Larmor formula Lenz law Liénard–Wiechert potential London equations Lorentz force Maxwell equations Maxwell tensor Poynting vector Synchrotron radiation Electrical network Alternating current 

In [14]:
faiss.write_index(index, "faiss_store/faiss_index.bin")

In [15]:
import json

with open("faiss_store/chunks.json", "w") as f:
    json.dump([chunk.page_content for chunk in all_splits], f)